In [1]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


Here we are going to learn Hyperprameterization, where we try multiple Hyperprameterization values to find the best suited value for our data for the model

In [1]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor)
import joblib


In [6]:
housing = datasets.fetch_california_housing()
x = housing.data
y = housing.target #to compare against

print(x.shape)

poly = PolynomialFeatures() #to make model smarter, we use optimization by introducing Polynomial Features, we take our existing features and expand them and generating way more data than before, which helps produce better results
#we improve model smartness from base 60% to 70% to 80% gradually
x = poly.fit_transform(x)

print(x.shape)

#reserving 20% data for testing/evaluation and remaining 80% will be used for training the model
# train_test_split this is going to return 4 parameter values
#def train_test_split(*arrays, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state = 432) #setting up value for random_state, shuffle data in a specific order each time we run the model for this data and can recreate the same workflow.

#Goal here is generating seveal instances of the same model with slightly diffrent parameter settings.
#max_iter is limits the tree structure
for i in [100,150, 200, 250, 300, 350, 400]:
  model = HistGradientBoostingRegressor(
    max_iter=i
  ) 
  model.fit(x_train, y_train) #order of the parameters matters
  #now that model is trained, time to evavluate the model
  y_pred = model.predict(x_test) #predicted results stored in y_pred, here y_pred is trying to predict y_test that has actual results of x_test
  #so to check how intelligent our model is we are going to compare y_test and y_pred, using some evaluation metrics
  r2 = r2_score(y_test, y_pred)

  print(i, r2) #it doesn't mean that our model made correct prediction in ~60% of cases - this is what accuracy score traces,
  #what it means is that our model undersands 60% of "WHY" - the housing prices go up and down, other 40% is mistry.
  #this is the baseline and now we will take this base line and keep improving it further by slite modifictions in code


(20640, 8)
(20640, 45)
100 0.838078761203522
150 0.8392758952727535
200 0.840078406836314
250 0.8441206190630384
300 0.8441883179809406
350 0.8480030284397888
400 0.8435964240714748


Here max_iter = 150 is best performing among [100,150, 200, 250, 300, 350, 400] many parameter values, after that it started declining.
But max_iter is not the only parameter we can optimize for the mode to produce better results,  there are more , so let's add Learning Rate parameter in nexsted loop below, I will leave the above for loop as is even though may not needed if I add another parameter below to evaluate the model outcome.


In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state = 432) 
for j in [0.1, 0.05, 0.001]:
    for i in [100,150, 200, 250, 300, 350, 400]:
        model = HistGradientBoostingRegressor(
        max_iter=i,
        learning_rate=j
        ) 
        model.fit(x_train, y_train) 
        y_pred = model.predict(x_test) 
        r2 = r2_score(y_test, y_pred)
        print(i, j, r2) 

100 0.1 0.8340386405112923
150 0.1 0.8393011061856264
200 0.1 0.8423758584927542
250 0.1 0.8409127585790824
300 0.1 0.8427346813024692
350 0.1 0.8404147613913298
400 0.1 0.8455174398871462
100 0.05 0.8250197579982457
150 0.05 0.8346385385431959
200 0.05 0.8371547716717712
250 0.05 0.840018417425382
300 0.05 0.8421646537896604
350 0.05 0.8413006136135843
400 0.05 0.8443730058057658
100 0.001 0.11999901080038211
150 0.001 0.17135964272380533
200 0.001 0.2209738340043187
250 0.001 0.2634989513445005
300 0.001 0.30274511643668356
350 0.001 0.3408735088563888
400 0.001 0.3728032186529514


Here best performing parameters are max_iter = 400 and learning_rate = 0.1 where r2 merics value is 0.0.8455174398871462 , r2 drastically declined after. So what we learnt here is Hyperprameterization.

So let's we are perfectly happy with our model performace with these Hyperprameters and don't want to optimize it further.
So we can save this model on system so that we don't have to train it again , save it using joblib library
Replace the best suited values in the code, max_iter = 400 and learning_rate = 0.1


In [17]:

model = HistGradientBoostingRegressor(
max_iter=400,
learning_rate=0.05 #chnaged , as performing better here
) 
model.fit(x_train, y_train) 
joblib.dump(model, "first_model_hgbr.joblib")
y_pred = model.predict(x_test) 
r2 = r2_score(y_test, y_pred)
print(r2) 

0.8423691971258761


Now file "first_model_hgbr.joblib" is generated by joblib , using joblib.dump(model, "first_model_hgbr.joblib") command.
If we want to use this model in this notebook or anyother workflow, we need to use joblib.load() command with paramters given below

In [18]:
local_model = joblib.load("first_model_hgbr.joblib")
y_pred = local_model.predict(x_test)  #predict() is now executed by using local_model , insted of model.
r2 = r2_score(y_test, y_pred)
print(r2) 

0.8423691971258761


If you note this local_model generated same r2 score as generated before.